# LeetCode 1992 — Find All Groups of Farmland bằng Python

Link đề: https://leetcode.com/problems/find-all-groups-of-farmland/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def findFarmland(self, land: List[List[int]]) -> List[List[int]]:
        ...
```

Lưới nhị phân, các vùng đất nông nghiệp (ô `1`) là **hình chữ nhật** và **không kề nhau**. Tìm góc trên-trái và dưới-phải của mỗi vùng. Notebook đi từ **BFS tìm thành phần liên thông + khung bao** (không cần giả thiết chữ nhật) → **DFS lặp** → **Union-Find** → **nhận diện góc trên-trái rồi đi sang phải/xuống dưới** (tận dụng giả thiết chữ nhật) → **ghép góc trên-trái với góc dưới-phải trong một lượt quét**.

## 1. Mục tiêu học

1. Duyệt lưới bằng **BFS/DFS lặp** để tìm thành phần liên thông và **khung bao** (min/max hàng, cột).
2. Tận dụng giả thiết "mọi nhóm là hình chữ nhật, không kề nhau" để có thuật toán **không cần đánh dấu** đã thăm.
3. Nhận diện **góc trên-trái**: ô `1` có ô trên và ô trái là `0` (hoặc ngoài lưới).
4. Kiểm thử bài "trả về theo thứ tự bất kỳ" bằng **validator** (so sánh sau khi sắp) và bộ sinh lưới ngẫu nhiên đúng ràng buộc.
5. Tránh các lỗi: kiểm thiếu điều kiện góc, sửa đổi đầu vào, đảo thứ tự tọa độ, đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `land` cỡ `m × n` (đánh số từ 0): `0` là rừng, `1` là đất nông nghiệp. Các **nhóm** là những vùng hình chữ nhật gồm toàn ô `1`; không có hai nhóm nào **kề nhau** (theo 4 hướng).

Với mỗi nhóm, trả về `[r1, c1, r2, c2]` — góc trên-trái `(r1, c1)` và góc dưới-phải `(r2, c2)`. Không có nhóm thì trả mảng rỗng. Trả về theo **thứ tự bất kỳ**.

**Ví dụ 1:** `land = [[1,0,0],[0,1,1],[0,1,1]]` → `[[0,0,0,0],[1,1,2,2]]`.

**Ví dụ 2:** `land = [[1,1],[1,1]]` → `[[0,0,1,1]]`.

**Ví dụ 3:** `land = [[0]]` → `[]`.

**Ràng buộc:** `1 ≤ m, n ≤ 300`, các nhóm là hình chữ nhật.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ([[1, 0, 0], [0, 1, 1], [0, 1, 1]], [[0, 0, 0, 0], [1, 1, 2, 2]]),
    ([[1, 1], [1, 1]], [[0, 0, 1, 1]]),
    ([[0]], []),
]

def draw(land):
    for row in land:
        print("   " + " ".join("█" if x else "·" for x in row))

for land, exp in examples:
    draw(land); print("   ->", exp)

   █ · ·
   · █ █
   · █ █
   -> [[0, 0, 0, 0], [1, 1, 2, 2]]
   █ █
   █ █
   -> [[0, 0, 1, 1]]
   ·
   -> []


## 3. Kiến thức nền

### 3.1 Bốn hướng kề và kiểm tra biên

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            yield nr, nc

print("hàng xóm của (0,0) trong lưới 3x3:", list(neighbors(0, 0, 3, 3)))
print("hàng xóm của (1,1) trong lưới 3x3:", list(neighbors(1, 1, 3, 3)))

hàng xóm của (0,0) trong lưới 3x3: [(1, 0), (0, 1)]
hàng xóm của (1,1) trong lưới 3x3: [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2 BFS với `deque` — tránh đệ quy sâu

Một nhóm có thể chứa `300 × 300 = 90 000` ô; DFS đệ quy sẽ vượt giới hạn đệ quy mặc định (1000) của Python. Dùng **BFS với hàng đợi** hoặc **DFS với stack tự quản**.

In [3]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit(), "| số ô tối đa một nhóm:", 300 * 300)

giới hạn đệ quy mặc định: 1000 | số ô tối đa một nhóm: 90000


### 3.3 Khung bao của tập ô

Với tập ô `S`: `r1 = min r`, `c1 = min c`, `r2 = max r`, `c2 = max c`. Nếu `S` là hình chữ nhật đầy thì `|S| = (r2 − r1 + 1)(c2 − c1 + 1)`.

### 3.4 Thứ tự bất kỳ ⇒ so sánh bằng `sorted`

In [4]:
a = [[1, 1, 2, 2], [0, 0, 0, 0]]
b = [[0, 0, 0, 0], [1, 1, 2, 2]]
print("a == b ?", a == b, "| sorted(a) == sorted(b) ?", sorted(a) == sorted(b))

a == b ? False | sorted(a) == sorted(b) ? True


## 4. Ý tưởng nền tảng / trực giác

Cách tổng quát: mỗi nhóm là một **thành phần liên thông** gồm các ô `1`; khung bao của nó là đáp án.

Nhưng đề cho thêm hai giả thiết mạnh: **chữ nhật** và **không kề nhau**. Khi đó:
- Ô `1` là **góc trên-trái** ⇔ ô phía trên và ô bên trái đều không phải `1`.
- Từ góc trên-trái, đi **sang phải** đến khi gặp `0`/biên ⇒ `c2`; đi **xuống dưới** ⇒ `r2`.

Ví dụ 1: ô `(1,1)` có trên `(0,1) = 0` và trái `(1,0) = 0` ⇒ góc trên-trái; sang phải tới cột 2, xuống tới hàng 2 ⇒ `[1,1,2,2]`.

In [5]:
land = examples[0][0]
m, n = len(land), len(land[0])
for r in range(m):
    print("   " + " ".join(
        ("◤" if land[r][c] and (r == 0 or not land[r - 1][c]) and (c == 0 or not land[r][c - 1]) else "█") if land[r][c] else "·"
        for c in range(n)))
print("   (◤ = góc trên-trái)")

   ◤ · ·
   · ◤ █
   · █ █
   (◤ = góc trên-trái)


## 5. Cách 1 — Brute Force / baseline: BFS tìm thành phần + khung bao

Không dùng giả thiết chữ nhật: duyệt mọi ô, gặp ô `1` chưa thăm thì BFS lấy cả thành phần, cập nhật min/max. Đúng cả khi các vùng có hình dạng bất kỳ (khi đó đáp án là khung bao).

In [6]:
def farmland_bfs(land):
    m, n = len(land), len(land[0])
    seen = [[False] * n for _ in range(m)]
    res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] == 1 and not seen[r][c]:
                seen[r][c] = True
                q = deque([(r, c)])
                r1, c1, r2, c2 = r, c, r, c
                while q:
                    x, y = q.popleft()
                    r1, c1, r2, c2 = min(r1, x), min(c1, y), max(r2, x), max(c2, y)
                    for nx, ny in neighbors(x, y, m, n):
                        if land[nx][ny] == 1 and not seen[nx][ny]:
                            seen[nx][ny] = True
                            q.append((nx, ny))
                res.append([r1, c1, r2, c2])
    return res

for land_, exp in examples:
    print(farmland_bfs(land_), "(kỳ vọng", exp, ")")

[[0, 0, 0, 0], [1, 1, 2, 2]] (kỳ vọng [[0, 0, 0, 0], [1, 1, 2, 2]] )
[[0, 0, 1, 1]] (kỳ vọng [[0, 0, 1, 1]] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(m · n)` thời gian, `O(m · n)` bộ nhớ (mảng `seen` + hàng đợi). Đã đủ nhanh! Nhưng tốn bộ nhớ phụ và không tận dụng cấu trúc chữ nhật.

**Nút thắt:** mảng `seen` `m × n` và việc thăm **mọi** ô trong nhóm bằng hàng đợi. Với giả thiết chữ nhật, ta chỉ cần **góc** và **hai cạnh**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề.** Trong lưới mà mọi nhóm là chữ nhật không kề nhau, ô `(r, c)` là góc trên-trái của một nhóm ⇔ `land[r][c] = 1`, `(r = 0 hoặc land[r−1][c] = 0)` và `(c = 0 hoặc land[r][c−1] = 0)`.

Vì: ô phía trên của một ô ở hàng đầu tiên của nhóm phải là `0` (ngược lại ô đó thuộc nhóm — trái giả thiết hàng đầu — hoặc thuộc nhóm khác kề nhau — trái giả thiết không kề). Tương tự cho cột trái. Ngược lại, ô không ở hàng đầu của nhóm có ô trên cũng thuộc nhóm (`= 1`).

**Thuật toán:** **nếu** ô là góc trên-trái **thì** đi sang phải để tìm `c2`, đi xuống để tìm `r2`; **ngược lại** bỏ qua. Không cần `seen`, không sửa đầu vào.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS lặp (stack) + đánh dấu bằng `set`

In [7]:
def farmland_dfs(land):
    m, n = len(land), len(land[0])
    seen = set()
    res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] and (r, c) not in seen:
                stack = [(r, c)]; seen.add((r, c))
                box = [r, c, r, c]
                while stack:
                    x, y = stack.pop()
                    box = [min(box[0], x), min(box[1], y), max(box[2], x), max(box[3], y)]
                    for nx, ny in neighbors(x, y, m, n):
                        if land[nx][ny] and (nx, ny) not in seen:
                            seen.add((nx, ny)); stack.append((nx, ny))
                res.append(box)
    return res

for land_, exp in examples:
    print(farmland_dfs(land_), "(kỳ vọng", exp, ")")

[[0, 0, 0, 0], [1, 1, 2, 2]] (kỳ vọng [[0, 0, 0, 0], [1, 1, 2, 2]] )
[[0, 0, 1, 1]] (kỳ vọng [[0, 0, 1, 1]] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(m · n)`; `set` các tuple tốn bộ nhớ hơn mảng bool nhưng gọn khi lưới thưa.

### 7.2 Cách 3 — Union-Find gom ô, rồi lấy khung bao mỗi gốc

In [8]:
def farmland_dsu(land):
    m, n = len(land), len(land[0])
    parent = list(range(m * n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    for r in range(m):
        for c in range(n):
            if land[r][c]:
                if r + 1 < m and land[r + 1][c]:
                    union(r * n + c, (r + 1) * n + c)
                if c + 1 < n and land[r][c + 1]:
                    union(r * n + c, r * n + c + 1)
    boxes = {}
    for r in range(m):
        for c in range(n):
            if land[r][c]:
                k = find(r * n + c)
                if k not in boxes:
                    boxes[k] = [r, c, r, c]
                else:
                    b = boxes[k]
                    b[0], b[1], b[2], b[3] = min(b[0], r), min(b[1], c), max(b[2], r), max(b[3], c)
    return list(boxes.values())

for land_, exp in examples:
    print(farmland_dsu(land_), "(kỳ vọng", exp, ")")

[[0, 0, 0, 0], [1, 1, 2, 2]] (kỳ vọng [[0, 0, 0, 0], [1, 1, 2, 2]] )
[[0, 0, 1, 1]] (kỳ vọng [[0, 0, 1, 1]] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(m · n · α)` (gần tuyến tính). Hữu ích khi lưới được **cập nhật dần** (xem LeetCode 305 Number of Islands II).

### 7.3 Cách 4 — Nhận diện góc trên-trái rồi đi theo hai cạnh

In [9]:
def farmland_corner(land):
    m, n = len(land), len(land[0])
    res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] == 0:
                continue
            if r > 0 and land[r - 1][c] == 1:
                continue                         # không phải hàng đầu của nhóm
            if c > 0 and land[r][c - 1] == 1:
                continue                         # không phải cột đầu của nhóm
            r2, c2 = r, c
            while r2 + 1 < m and land[r2 + 1][c] == 1:
                r2 += 1
            while c2 + 1 < n and land[r][c2 + 1] == 1:
                c2 += 1
            res.append([r, c, r2, c2])
    return res

for land_, exp in examples:
    print(farmland_corner(land_), "(kỳ vọng", exp, ")")

[[0, 0, 0, 0], [1, 1, 2, 2]] (kỳ vọng [[0, 0, 0, 0], [1, 1, 2, 2]] )
[[0, 0, 1, 1]] (kỳ vọng [[0, 0, 1, 1]] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(m · n)` cho lượt quét + tổng chu vi các nhóm cho các bước đi cạnh (`≤ m · n`). Bộ nhớ phụ `O(1)` ngoài kết quả.

### 7.4 Cách 5 — Một lượt quét: ghép góc trên-trái với góc dưới-phải

Góc **dưới-phải** là ô `1` có ô dưới và ô phải không phải `1`. Quét theo hàng: góc trên-trái của các nhóm xuất hiện theo thứ tự; khi gặp **góc dưới-phải** `(r, c)`, nhóm tương ứng là nhóm có góc trên-trái ở **cùng cột trái** — ta tra bằng từ điển `open_by_right_col`: khi gặp góc trên-trái `(r1, c1)`, đi sang phải lấy `c2` (cạnh trên), lưu `open[c2] = (r1, c1)`; khi gặp góc dưới-phải ở cột `c2`, lấy `open.pop(c2)`.

Nhóm có cột phải `c2` là duy nhất tại mỗi thời điểm "đang mở" (hai nhóm mở cùng lúc có cùng cột phải sẽ chồng nhau theo cột — không thể vì chúng chồng hàng nên phải tách cột).

In [10]:
def farmland_one_pass(land):
    m, n = len(land), len(land[0])
    open_by_right = {}
    res = []
    for r in range(m):
        for c in range(n):
            if not land[r][c]:
                continue
            up = r > 0 and land[r - 1][c]
            left = c > 0 and land[r][c - 1]
            if not up and not left:                         # góc trên-trái
                c2 = c
                while c2 + 1 < n and land[r][c2 + 1]:
                    c2 += 1
                open_by_right[c2] = (r, c)
            down = r + 1 < m and land[r + 1][c]
            right = c + 1 < n and land[r][c + 1]
            if not down and not right:                      # góc dưới-phải
                r1, c1 = open_by_right.pop(c)
                res.append([r1, c1, r, c])
    return res

for land_, exp in examples:
    print(farmland_one_pass(land_), "(kỳ vọng", exp, ")")

[[0, 0, 0, 0], [1, 1, 2, 2]] (kỳ vọng [[0, 0, 0, 0], [1, 1, 2, 2]] )
[[0, 0, 1, 1]] (kỳ vọng [[0, 0, 1, 1]] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(m · n)` + chu vi trên; bộ nhớ `O(số nhóm đang mở) ≤ O(n)`.

## 8. TRACE từng bước

### 8.1 Trace cách 4 trên ví dụ 1

In [11]:
def trace_corner(land):
    m, n = len(land), len(land[0])
    res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] == 0:
                continue
            up = r > 0 and land[r - 1][c] == 1
            left = c > 0 and land[r][c - 1] == 1
            if up or left:
                print(f"  ({r},{c}) = 1 nhưng {'trên' if up else 'trái'} cũng là 1 -> không phải góc")
                continue
            r2, c2 = r, c
            while r2 + 1 < m and land[r2 + 1][c] == 1: r2 += 1
            while c2 + 1 < n and land[r][c2 + 1] == 1: c2 += 1
            print(f"  ({r},{c}) là góc trên-trái: đi xuống tới hàng {r2}, sang phải tới cột {c2} -> {[r, c, r2, c2]}")
            res.append([r, c, r2, c2])
    return res

print("=>", trace_corner(examples[0][0]))

  (0,0) là góc trên-trái: đi xuống tới hàng 0, sang phải tới cột 0 -> [0, 0, 0, 0]
  (1,1) là góc trên-trái: đi xuống tới hàng 2, sang phải tới cột 2 -> [1, 1, 2, 2]
  (1,2) = 1 nhưng trái cũng là 1 -> không phải góc
  (2,1) = 1 nhưng trên cũng là 1 -> không phải góc
  (2,2) = 1 nhưng trên cũng là 1 -> không phải góc
=> [[0, 0, 0, 0], [1, 1, 2, 2]]


### 8.2 Trace cách 5 trên một lưới tự dựng (đã đối chiếu BFS — đối chiếu lại đề gốc)

In [12]:
def trace_one_pass(land):
    m, n = len(land), len(land[0])
    open_by_right, res = {}, []
    for r in range(m):
        for c in range(n):
            if not land[r][c]:
                continue
            up = r > 0 and land[r - 1][c]; left = c > 0 and land[r][c - 1]
            if not up and not left:
                c2 = c
                while c2 + 1 < n and land[r][c2 + 1]: c2 += 1
                open_by_right[c2] = (r, c)
                print(f"  ({r},{c}) góc trên-trái, cạnh trên tới cột {c2}: mở nhóm, open = {open_by_right}")
            down = r + 1 < m and land[r + 1][c]; right = c + 1 < n and land[r][c + 1]
            if not down and not right:
                r1, c1 = open_by_right.pop(c)
                res.append([r1, c1, r, c])
                print(f"  ({r},{c}) góc dưới-phải: đóng nhóm {[r1, c1, r, c]}, open = {open_by_right}")
    return res

grid = [
    [1, 1, 0, 1],
    [1, 1, 0, 1],
    [0, 0, 0, 1],
    [1, 0, 1, 0],
]
draw(grid)
got = trace_one_pass(grid)
print("=>", got, "| BFS:", farmland_bfs(grid))

   █ █ · █
   █ █ · █
   · · · █
   █ · █ ·
  (0,0) góc trên-trái, cạnh trên tới cột 1: mở nhóm, open = {1: (0, 0)}
  (0,3) góc trên-trái, cạnh trên tới cột 3: mở nhóm, open = {1: (0, 0), 3: (0, 3)}
  (1,1) góc dưới-phải: đóng nhóm [0, 0, 1, 1], open = {3: (0, 3)}
  (2,3) góc dưới-phải: đóng nhóm [0, 3, 2, 3], open = {}
  (3,0) góc trên-trái, cạnh trên tới cột 0: mở nhóm, open = {0: (3, 0)}
  (3,0) góc dưới-phải: đóng nhóm [3, 0, 3, 0], open = {}
  (3,2) góc trên-trái, cạnh trên tới cột 2: mở nhóm, open = {2: (3, 2)}
  (3,2) góc dưới-phải: đóng nhóm [3, 2, 3, 2], open = {}
=> [[0, 0, 1, 1], [0, 3, 2, 3], [3, 0, 3, 0], [3, 2, 3, 2]] | BFS: [[0, 0, 1, 1], [0, 3, 2, 3], [3, 0, 3, 0], [3, 2, 3, 2]]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def findFarmland(self, land: List[List[int]]) -> List[List[int]]:
        m, n = len(land), len(land[0])
        res = []
        for r in range(m):
            row = land[r]
            for c in range(n):
                if row[c] != 1:
                    continue
                if (r > 0 and land[r - 1][c] == 1) or (c > 0 and row[c - 1] == 1):
                    continue                    # không phải góc trên-trái
                r2 = r
                while r2 + 1 < m and land[r2 + 1][c] == 1:
                    r2 += 1
                c2 = c
                while c2 + 1 < n and row[c2 + 1] == 1:
                    c2 += 1
                res.append([r, c, r2, c2])
        return res

def same(a, b):
    return sorted(map(list, a)) == sorted(map(list, b))

for land_, exp in examples:
    assert same(Solution().findFarmland(land_), exp)
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cách 1–3** tìm đúng các thành phần liên thông 4 hướng của ô `1` (BFS/DFS/Union-Find chuẩn). Vì các nhóm **không kề nhau**, mỗi thành phần đúng bằng một nhóm; nhóm là chữ nhật nên khung bao chính là `[r1, c1, r2, c2]` của nhóm.

**Cách 4.** (i) Mỗi nhóm có **đúng một** góc trên-trái thỏa điều kiện (bổ đề mục 6): ô `(r1, c1)` thỏa; mọi ô khác của nhóm hoặc có ô trên thuộc nhóm (nếu không ở hàng `r1`) hoặc có ô trái thuộc nhóm (nếu ở hàng `r1` nhưng cột `> c1`). (ii) Đi xuống từ `(r1, c1)` qua các ô `1` dừng đúng ở `r2`: các ô `(r, c1)` với `r1 ≤ r ≤ r2` thuộc nhóm; ô `(r2 + 1, c1)` nếu là `1` thì thuộc nhóm khác kề với nhóm này — trái giả thiết. Tương tự cho `c2`. ⇒ mỗi nhóm được báo cáo đúng một lần với tọa độ đúng.

**Cách 5.** Góc dưới-phải của mỗi nhóm cũng duy nhất. Khi quét theo hàng, góc trên-trái `(r1, c1)` luôn được gặp **trước** góc dưới-phải `(r2, c2)`. Trong khoảng thời gian một nhóm "đang mở" (hàng `r1..r2`), không nhóm nào khác mở có cùng cột phải `c2` (chúng sẽ chồng lên nhau tại cột `c2` ở các hàng chung). ⇒ `open_by_right[c2]` xác định đúng nhóm.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space phụ |
|---|---|---|
| 1. BFS | `O(mn)` | `O(mn)` |
| 2. DFS lặp + set | `O(mn)` | `O(mn)` |
| 3. Union-Find | `O(mn · α)` | `O(mn)` |
| 4. Góc trên-trái | `O(mn)` | `O(1)` |
| 5. Một lượt ghép góc | `O(mn)` | `O(n)` |

In [14]:
def gen_land(m, n, rnd, tries=200, max_side=8):
    """Sinh lưới m x n với các chữ nhật toàn 1, không chồng và không kề nhau (có viền 0)."""
    land = [[0] * n for _ in range(m)]
    blocked = [[False] * n for _ in range(m)]
    rects = []
    for _ in range(tries):
        h, w = rnd.randint(1, min(max_side, m)), rnd.randint(1, min(max_side, n))
        r1, c1 = rnd.randint(0, m - h), rnd.randint(0, n - w)
        r2, c2 = r1 + h - 1, c1 + w - 1
        if any(blocked[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1)):
            continue
        for r in range(max(0, r1 - 1), min(m, r2 + 2)):
            for c in range(max(0, c1 - 1), min(n, c2 + 2)):
                blocked[r][c] = True                 # chặn cả viền để không kề nhau
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                land[r][c] = 1
        rects.append([r1, c1, r2, c2])
    return land, rects

rnd = random.Random(1992)
big, big_rects = gen_land(300, 300, rnd, tries=3000, max_side=20)
full = [[1] * 300 for _ in range(300)]
for name, fn in [("bfs", farmland_bfs), ("dfs", farmland_dfs), ("dsu", farmland_dsu), ("corner", farmland_corner), ("one_pass", farmland_one_pass), ("Solution", Solution().findFarmland)]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    t1 = time.perf_counter(); fn(full); dt2 = time.perf_counter() - t1
    print(f"{name:<9}: {len(r):>4} nhóm, lưới ngẫu nhiên {dt * 1000:7.1f} ms | lưới toàn 1: {dt2 * 1000:7.1f} ms")

bfs      :  527 nhóm, lưới ngẫu nhiên    28.5 ms | lưới toàn 1:    65.8 ms


dfs      :  527 nhóm, lưới ngẫu nhiên    31.9 ms | lưới toàn 1:    94.2 ms


dsu      :  527 nhóm, lưới ngẫu nhiên    29.2 ms | lưới toàn 1:    90.9 ms
corner   :  527 nhóm, lưới ngẫu nhiên     3.6 ms | lưới toàn 1:     4.9 ms
one_pass :  527 nhóm, lưới ngẫu nhiên     5.0 ms | lưới toàn 1:    10.5 ms
Solution :  527 nhóm, lưới ngẫu nhiên     3.5 ms | lưới toàn 1:     4.5 ms


## 12. Bộ test

In [15]:
ALL = {
    "bfs": farmland_bfs,
    "dfs": farmland_dfs,
    "dsu": farmland_dsu,
    "corner": farmland_corner,
    "one_pass": farmland_one_pass,
    "Solution": lambda g: Solution().findFarmland(g),
}

def is_valid_answer(land, ans):
    """Validator độc lập: mỗi hình chữ nhật toàn 1, viền ngoài toàn 0, phủ đúng mọi ô 1, không trùng."""
    m, n = len(land), len(land[0])
    cover = [[0] * n for _ in range(m)]
    for r1, c1, r2, c2 in ans:
        if not (0 <= r1 <= r2 < m and 0 <= c1 <= c2 < n):
            return False
        for r in range(r1, r2 + 1):
            for c in range(c1, c2 + 1):
                if land[r][c] != 1:
                    return False
                cover[r][c] += 1
        for r in range(r1, r2 + 1):
            if (c1 > 0 and land[r][c1 - 1]) or (c2 + 1 < n and land[r][c2 + 1]):
                return False
        for c in range(c1, c2 + 1):
            if (r1 > 0 and land[r1 - 1][c]) or (r2 + 1 < m and land[r2 + 1][c]):
                return False
    return all(cover[r][c] == land[r][c] for r in range(m) for c in range(n))

def check(land, exp=None):
    snapshot = [row[:] for row in land]
    ref = farmland_bfs(land)
    assert is_valid_answer(land, ref)
    for name, fn in ALL.items():
        got = fn(land)
        assert same(got, ref) and is_valid_answer(land, got), (name, land, got, ref)
        assert land == snapshot, name + " đã sửa đầu vào!"
    if exp is not None:
        assert same(ref, exp)
    return ref

for land_, exp in examples:
    check(land_, exp)
print("Ví dụ đề: OK")

check([[1]], [[0, 0, 0, 0]])
check([[1, 1, 1]], [[0, 0, 0, 2]])
check([[1], [1], [1]], [[0, 0, 2, 0]])
check([[1, 0, 1], [0, 0, 0], [1, 0, 1]], [[0, 0, 0, 0], [0, 2, 0, 2], [2, 0, 2, 0], [2, 2, 2, 2]])
check([[0, 0], [0, 0]], [])
check(full, [[0, 0, 299, 299]])
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(400):
    m, n = rnd.randint(1, 12), rnd.randint(1, 12)
    land_, rects = gen_land(m, n, rnd, tries=rnd.randint(0, 30), max_side=5)
    check(land_, rects)
print("400 lưới ngẫu nhiên nhỏ (so với danh sách chữ nhật đã sinh): OK")

for _ in range(3):
    land_, rects = gen_land(300, 300, rnd, tries=2000, max_side=25)
    check(land_, rects)
print("3 lưới 300x300 ngẫu nhiên: OK")

Ví dụ đề: OK


Edge case: OK
400 lưới ngẫu nhiên nhỏ (so với danh sách chữ nhật đã sinh): OK


3 lưới 300x300 ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Điều kiện góc chỉ kiểm ô phía trên

In [16]:
def farmland_bug_up_only(land):
    m, n = len(land), len(land[0]); res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] and (r == 0 or not land[r - 1][c]):      # SAI: quên kiểm ô bên trái
                r2, c2 = r, c
                while r2 + 1 < m and land[r2 + 1][c]: r2 += 1
                while c2 + 1 < n and land[r][c2 + 1]: c2 += 1
                res.append([r, c, r2, c2])
    return res

land_ = [[1, 1, 1]]
print("sai :", farmland_bug_up_only(land_), "| đúng:", Solution().findFarmland(land_))

sai : [[0, 0, 0, 2], [0, 1, 0, 2], [0, 2, 0, 2]] | đúng: [[0, 0, 0, 2]]


**Vì sao sai:** mọi ô ở hàng đầu tiên của nhóm đều có ô trên là `0`, nên mỗi ô cạnh trên bị coi là một góc ⇒ thêm các "nhóm" con sai. **Sửa:** kiểm cả `land[r][c−1]`.

### Lỗi 2 — "Đánh dấu đã thăm" bằng cách gán `0` vào chính `land`

In [17]:
def farmland_bug_mutate(land):
    m, n = len(land), len(land[0]); res = []
    for r in range(m):
        for c in range(n):
            if land[r][c]:
                r2, c2 = r, c
                while r2 + 1 < m and land[r2 + 1][c]: r2 += 1
                while c2 + 1 < n and land[r][c2 + 1]: c2 += 1
                for x in range(r, r2 + 1):
                    for y in range(c, c2 + 1):
                        land[x][y] = 0                   # phá hủy đầu vào
                res.append([r, c, r2, c2])
    return res

land_ = [[1, 0, 0], [0, 1, 1], [0, 1, 1]]
print("lần 1:", farmland_bug_mutate(land_))
print("lần 2 trên cùng biến:", farmland_bug_mutate(land_), "| lưới sau khi gọi:", land_)

lần 1: [[0, 0, 0, 0], [1, 1, 2, 2]]
lần 2 trên cùng biến: [] | lưới sau khi gọi: [[0, 0, 0], [0, 0, 0], [0, 0, 0]]


**Vì sao sai:** kết quả lần đầu đúng, nhưng **đầu vào bị phá** — nơi gọi dùng lại `land` (in ra, kiểm tra, gọi lần nữa) sẽ nhận kết quả sai. LeetCode chấp nhận, nhưng trong dự án thật đây là **tác dụng phụ** khó phát hiện. **Sửa:** dùng góc trên-trái (không cần đánh dấu) hoặc mảng `seen` riêng.

### Lỗi 3 — Đảo thứ tự tọa độ `[c1, r1, c2, r2]`

In [18]:
def farmland_bug_order(land):
    return [[c1, r1, c2, r2] for r1, c1, r2, c2 in Solution().findFarmland(land)]

land_ = [[1, 1, 1], [1, 1, 1]]
print("sai :", farmland_bug_order(land_), "| đúng:", Solution().findFarmland(land_))

sai : [[0, 0, 2, 1]] | đúng: [[0, 0, 1, 2]]


**Vì sao sai:** đề yêu cầu `[r1, c1, r2, c2]` (hàng trước, cột sau). Với nhóm vuông đối xứng quanh đường chéo thì lỗi bị **che khuất** (ví dụ 2 vẫn đúng!) — luôn test nhóm **không vuông**.

### Lỗi 4 — Tìm `c2` bằng cách đi sang phải từ **hàng dưới cùng** nhưng `r2` bị tính sai do đi xuống ở cột sai

In [19]:
def farmland_bug_walk(land):
    m, n = len(land), len(land[0]); res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] and (r == 0 or not land[r - 1][c]) and (c == 0 or not land[r][c - 1]):
                c2 = c
                while c2 + 1 < n and land[r][c2 + 1]: c2 += 1
                r2 = r
                while r2 + 1 < m and land[r2 + 1][c2 + 1 if c2 + 1 < n else c2]: r2 += 1   # SAI: dò ở cột NGOÀI nhóm
                res.append([r, c, r2, c2])
    return res

land_ = [[1, 1, 0], [1, 1, 0], [0, 0, 0]]
print("sai :", farmland_bug_walk(land_), "| đúng:", Solution().findFarmland(land_))

sai : [[0, 0, 0, 1]] | đúng: [[0, 0, 1, 1]]


**Vì sao sai:** lỗi "lệch một" khi chọn cột để dò xuống — cột `c2 + 1` nằm **ngoài** nhóm (toàn `0`), nên `r2` dừng ngay ở `r`. **Sửa:** dò xuống ở cột `c` (hoặc `c2`) — một cột **trong** nhóm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Lưới + "vùng liên thông" ⇒ BFS/DFS lặp hoặc Union-Find; **không** dùng DFS đệ quy cho lưới `300 × 300`.
- Đề cho **hình dạng đặc biệt** (chữ nhật, không kề) ⇒ tìm **điểm đặc trưng** (góc) để tránh duyệt cả vùng.
- "Trả về thứ tự bất kỳ" ⇒ test bằng `sorted` hoặc validator.
- Luôn test hình **không vuông** để bắt lỗi đảo hàng/cột.
- Muốn không sửa đầu vào: mảng `seen` riêng hoặc thuật toán không cần đánh dấu.

## 15. Biến thể của bài

1. **LeetCode 200 — Number of Islands:** đếm thành phần, hình dạng tùy ý.
2. **LeetCode 695 — Max Area of Island:** diện tích lớn nhất.
3. **LeetCode 1905 — Count Sub Islands.**
4. **Vùng không chắc là chữ nhật:** kiểm tra từng thành phần có phải chữ nhật không (`|S| == diện tích khung bao`) — bài tập 5.
5. **LeetCode 305 — Number of Islands II:** thêm ô dần, đếm online bằng Union-Find.

In [20]:
def rectangle_components(land):
    """Biến thể: với lưới bất kỳ, liệt kê thành phần và cho biết có phải chữ nhật đầy không."""
    m, n = len(land), len(land[0])
    seen = [[False] * n for _ in range(m)]
    out = []
    for r in range(m):
        for c in range(n):
            if land[r][c] and not seen[r][c]:
                seen[r][c] = True; q = deque([(r, c)]); cells = 0; box = [r, c, r, c]
                while q:
                    x, y = q.popleft(); cells += 1
                    box = [min(box[0], x), min(box[1], y), max(box[2], x), max(box[3], y)]
                    for nx, ny in neighbors(x, y, m, n):
                        if land[nx][ny] and not seen[nx][ny]:
                            seen[nx][ny] = True; q.append((nx, ny))
                area = (box[2] - box[0] + 1) * (box[3] - box[1] + 1)
                out.append((box, cells == area))
    return out

print(rectangle_components([[1, 1, 0], [1, 0, 0], [0, 0, 1]]))

[([0, 0, 1, 1], False), ([2, 2, 2, 2], True)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | thành phần + khung bao | `O(mn)` | `O(mn)` | tổng quát, đối chiếu |
| 2. DFS lặp | như BFS, stack | `O(mn)` | `O(mn)` | tương tự |
| 3. Union-Find | gom ô kề | `O(mn α)` | `O(mn)` | cập nhật online |
| 4. Góc trên-trái | đi hai cạnh | `O(mn)` | `O(1)` | **nộp bài** |
| 5. Ghép góc 1 lượt | từ điển cột phải | `O(mn)` | `O(n)` | quét dạng luồng theo hàng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().findFarmland([[1,1,0,0],[0,0,0,1],[1,0,0,1]])` (sắp kết quả)?

**Bài 2 (tính tay):** Trong lưới ở bài 1, ô nào là **góc dưới-phải**?

**Bài 3 (điền TODO):** Hoàn thiện `farmland_areas(land)` — trả về danh sách diện tích các nhóm, **sắp giảm dần**, dùng kết quả của `findFarmland`.

**Bài 4 (tự viết lại):** Viết lại cách 4 nhưng dò từ **góc dưới-phải** (ô có dưới và phải là `0`), đi lên và sang trái.

**Bài 5 (mini challenge):** Với lưới **bất kỳ** (không đảm bảo chữ nhật), viết `all_rectangles(land)` trả `True` nếu mọi thành phần là chữ nhật **và** đề bài 1992 áp dụng được; kiểm bằng cách so `farmland_corner` với `farmland_bfs` trên các lưới ngẫu nhiên — chúng **trùng nhau mọi khi** `all_rectangles` đúng.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def farmland_areas(land):
    # TODO: (r2 - r1 + 1) * (c2 - c1 + 1) cho mỗi nhóm, sắp giảm
    pass

# Bài 4 — TODO
def farmland_from_bottom_right(land):
    # TODO: ô 1 có (r+1, c) và (r, c+1) không phải 1 -> đi lên, đi sang trái
    pass

# Bài 5 — TODO
def all_rectangles(land):
    # TODO: BFS từng thành phần, so số ô với diện tích khung bao
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Có ba nhóm.
- Mức 2: `(0,0)-(0,1)`, `(1,3)-(2,3)`, `(2,0)`.
- Mức 3: `[[0,0,0,1],[1,3,2,3],[2,0,2,0]]`.

**Bài 2**
- Mức 1: Mỗi nhóm có đúng một góc dưới-phải.
- Mức 2: Ô `(0,1)`, `(2,3)`, `(2,0)`.
- Mức 3: Kiểm bằng điều kiện dưới/phải là `0` hoặc ngoài lưới.

**Bài 3**
- Mức 1: Tính diện tích từ tọa độ.
- Mức 2: `sorted(..., reverse=True)`.
- Mức 3: Đối chiếu với số ô của từng thành phần BFS.

**Bài 4**
- Mức 1: Đối xứng với cách 4.
- Mức 2: `while r1 - 1 >= 0 and land[r1 - 1][c]`.
- Mức 3: Kết quả trùng cách 4 sau khi sắp.

**Bài 5**
- Mức 1: Dùng `rectangle_components` ở mục 15.
- Mức 2: Còn phải kiểm "không kề" — tự động đúng vì thành phần liên thông khác nhau thì không kề.
- Mức 3: Sinh lưới ngẫu nhiên tùy ý, lọc lưới thỏa, so hai thuật toán.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
b1 = [[1, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 1]]
r1 = sorted(Solution().findFarmland(b1))
print("Bài 1:", r1); assert r1 == [[0, 0, 0, 1], [1, 3, 2, 3], [2, 0, 2, 0]]

# Bài 2
m, n = 3, 4
br = [(r, c) for r in range(m) for c in range(n)
      if b1[r][c] and not (r + 1 < m and b1[r + 1][c]) and not (c + 1 < n and b1[r][c + 1])]
print("Bài 2:", br); assert sorted(br) == [(0, 1), (2, 0), (2, 3)]

# Bài 3
def farmland_areas_sol(land):
    return sorted(((r2 - r1 + 1) * (c2 - c1 + 1) for r1, c1, r2, c2 in Solution().findFarmland(land)), reverse=True)

rnd = random.Random(3)
for _ in range(200):
    g, rects = gen_land(rnd.randint(1, 10), rnd.randint(1, 10), rnd, tries=20, max_side=4)
    sizes = sorted((b[2] - b[0] + 1) * (b[3] - b[1] + 1) for b, _ in rectangle_components(g)) if any(map(any, g)) else []
    assert farmland_areas_sol(g) == sorted(sizes, reverse=True)
print("Bài 3 OK, ví dụ:", farmland_areas_sol(b1))

Bài 1: [[0, 0, 0, 1], [1, 3, 2, 3], [2, 0, 2, 0]]
Bài 2: [(0, 1), (2, 0), (2, 3)]
Bài 3 OK, ví dụ: [2, 2, 1]


In [23]:
# Bài 4
def farmland_from_bottom_right_sol(land):
    m, n = len(land), len(land[0]); res = []
    for r in range(m):
        for c in range(n):
            if land[r][c] and not (r + 1 < m and land[r + 1][c]) and not (c + 1 < n and land[r][c + 1]):
                r1, c1 = r, c
                while r1 - 1 >= 0 and land[r1 - 1][c]: r1 -= 1
                while c1 - 1 >= 0 and land[r][c1 - 1]: c1 -= 1
                res.append([r1, c1, r, c])
    return res

# Bài 5
def all_rectangles_sol(land):
    return all(ok for _, ok in rectangle_components(land)) if any(map(any, land)) else True

rnd = random.Random(5)
agree_when_rect = 0; rect_count = 0
for _ in range(1500):
    m, n = rnd.randint(1, 6), rnd.randint(1, 6)
    g = [[1 if rnd.random() < 0.45 else 0 for _ in range(n)] for _ in range(m)]
    if all_rectangles_sol(g):
        rect_count += 1
        assert same(farmland_corner(g), farmland_bfs(g))
        assert same(farmland_from_bottom_right_sol(g), farmland_bfs(g))
        agree_when_rect += 1
for _ in range(200):
    g, rects = gen_land(rnd.randint(1, 12), rnd.randint(1, 12), rnd, tries=25, max_side=5)
    assert all_rectangles_sol(g) and same(farmland_from_bottom_right_sol(g), rects)
print(f"Bài 4, Bài 5 OK: {rect_count} lưới ngẫu nhiên thỏa giả thiết, cách góc trùng BFS ở cả {agree_when_rect}")
print("ví dụ lưới không chữ nhật:", all_rectangles_sol([[1, 1], [1, 0]]))

Bài 4, Bài 5 OK: 785 lưới ngẫu nhiên thỏa giả thiết, cách góc trùng BFS ở cả 785
ví dụ lưới không chữ nhật: False


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template góc trên-trái:**

```python
for r in range(m):
    for c in range(n):
        if land[r][c] != 1: continue
        if (r and land[r-1][c]) or (c and land[r][c-1]): continue
        r2, c2 = r, c
        while r2 + 1 < m and land[r2+1][c]: r2 += 1
        while c2 + 1 < n and land[r][c2+1]: c2 += 1
        res.append([r, c, r2, c2])
```

**Checklist:**
- [ ] Kiểm **cả** ô trên và ô trái khi nhận diện góc?
- [ ] Không sửa `land` (hoặc chấp nhận có ý thức)?
- [ ] Thứ tự `[r1, c1, r2, c2]`?
- [ ] Không dùng DFS đệ quy cho lưới lớn?
- [ ] Đã test nhóm không vuông, nhóm 1 ô, lưới toàn 0, lưới toàn 1?
- [ ] Chữ ký đúng: `def findFarmland(self, land: List[List[int]]) -> List[List[int]]`?